
# Actual `ymcirc` / `pyclebsch` SU(3) \(O(y^4)\) capability audit

This notebook does not recheck our own \(H_4\) certificate. It clones and executes the current public `ymcirc` and `pyclebsch` repositories.

It performs four concrete tasks:

1. audits the truncations and dimensional data actually shipped by `ymcirc`;
2. installs a ten-irrep `Y4` truncation map needed by the certified fourth-order paths;
3. uses `pyclebsch.find_direct_sum` to independently verify all 30 required SU(3) fusion edges;
4. optionally computes and orthogonality-checks the Clebsch–Gordan data for all 18 unique source×token products.

The final output is a machine-readable capability report stating exactly which microscopic layers are reproducible with the current public software and which data must still be generated before a full three-dimensional cube calculation can run.

Use a **standard Colab CPU runtime**. GPU acceleration is not used.


In [ ]:

from pathlib import Path
import os, shutil, subprocess, sys

ROOT = Path("/content")
YMCIRC_DIR = ROOT / "ymcirc_repo"
PYCLEBSCH_DIR = ROOT / "pyclebsch_repo"

def clone_at(url, dest, commit):
    if not dest.exists():
        subprocess.run(["git", "clone", "--quiet", url, str(dest)], check=True)
    subprocess.run(["git", "-C", str(dest), "fetch", "--quiet", "origin"], check=True)
    subprocess.run(["git", "-C", str(dest), "checkout", "--quiet", commit], check=True)

clone_at("https://github.com/hepqis-uiuc/ymcirc.git", YMCIRC_DIR, "f2073f6bd61758a8ab56975ac53c5be340e8cffc")
clone_at("https://github.com/hepqis-uiuc/pyclebsch.git", PYCLEBSCH_DIR, "35e3926b07761da5bfbe573fd6af74e64a0c1a82")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PYCLEBSCH_DIR / "NOTE_MISC_requirements.txt")],
    check=True,
)
# ymcirc imports Qiskit even for static capability inspection.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(YMCIRC_DIR / "NOTE_MISC_requirements.txt")],
    check=True,
)

for repo in (YMCIRC_DIR, PYCLEBSCH_DIR):
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))

print("ymcirc commit   :", subprocess.check_output(
    ["git", "-C", str(YMCIRC_DIR), "rev-parse", "HEAD"], text=True
).strip())
print("pyclebsch commit:", subprocess.check_output(
    ["git", "-C", str(PYCLEBSCH_DIR), "rev-parse", "HEAD"], text=True
).strip())


In [ ]:

import json, time
from collections import Counter, defaultdict
from pathlib import Path

import ymcirc.conventions as yc
import pyclebsch.su_n_operators as ops
import pyclebsch.cgc as cgc

print("Imported ymcirc and pyclebsch successfully.")


## Audit the actual public `ymcirc` data

In [ ]:

def dynkin_to_iweight(dynkin):
    p, q = map(int, dynkin)
    return (p + q, q, 0)

REQUIRED_DYNKIN = {
    (0,0): "1",
    (1,0): "3",
    (0,1): "3bar",
    (2,0): "6",
    (0,2): "6bar",
    (1,1): "8",
    (3,0): "10",
    (0,3): "10bar",
    (2,1): "15",
    (1,2): "15bar",
}
REQUIRED_IWEIGHTS = {
    dynkin_to_iweight(key): value
    for key, value in REQUIRED_DYNKIN.items()
}

current_truncations = {
    name: {tuple(irrep): bits for irrep, bits in bitmap.items()}
    for name, bitmap in yc.IRREP_TRUNCATIONS.items()
}
current_irreps = set().union(*(set(x) for x in current_truncations.values()))
missing_iweights = {
    irrep: name for irrep, name in REQUIRED_IWEIGHTS.items()
    if irrep not in current_irreps
}

available_hamiltonian_data = {
    dim: sorted(entries)
    for dim, entries in yc._HAMILTONIAN_DATA_FILE_PATHS.items()
}
available_plaquette_data = {
    dim: sorted(entries)
    for dim, entries in yc._PLAQUETTE_STATES_DATA_FILE_PATHS.items()
}

print("Public ymcirc truncations:")
for name, bitmap in current_truncations.items():
    print(" ", name, sorted(bitmap))

print("\nPublic magnetic-Hamiltonian data:", available_hamiltonian_data)
print("Public plaquette-state data      :", available_plaquette_data)
print("Missing certified Y4 irreps      :", missing_iweights)

assert "d=3" not in available_hamiltonian_data
assert "d=3" not in available_plaquette_data
assert set(current_truncations["T2"]) == {
    (0,0,0), (1,0,0), (1,1,0), (2,0,0), (2,2,0), (2,1,0)
}
assert set(missing_iweights) == {
    (3,0,0), (3,3,0), (3,1,0), (3,2,0)
}

print("\nAUDIT PASS: current public ymcirc lacks d=3 dictionaries and the 10/10bar/15/15bar channels.")


## Install the certified ten-irrep truncation map

In [ ]:

# Install a deterministic four-bit encoding for the ten required irreps.
# This makes the representation register available to ymcirc, but does not
# fabricate the missing d=3 site-singlet or magnetic matrix-element data.

Y4_ORDER = [
    (0,0,0),  # 1
    (1,0,0),  # 3
    (1,1,0),  # 3bar
    (2,0,0),  # 6
    (2,2,0),  # 6bar
    (2,1,0),  # 8
    (3,0,0),  # 10
    (3,3,0),  # 10bar
    (3,1,0),  # 15
    (3,2,0),  # 15bar
]
Y4_BITMAP = {
    irrep: format(index, "04b")
    for index, irrep in enumerate(Y4_ORDER)
}
yc.IRREP_TRUNCATIONS["Y4"] = Y4_BITMAP

assert len(Y4_BITMAP) == 10
assert set(Y4_BITMAP) == set(REQUIRED_IWEIGHTS)

print("Installed in-memory ymcirc Y4 truncation:")
print(json.dumps({str(k): v for k, v in Y4_BITMAP.items()}, indent=2))


## Independently verify all 30 fusion edges

In [ ]:

CERTIFIED_EDGES = json.loads('[{"edge_id":"FTI-01","source_dynkin":[0,0],"token":-1,"target_dynkin":[0,1],"target_dimension":3},{"edge_id":"FTI-02","source_dynkin":[0,0],"token":1,"target_dynkin":[1,0],"target_dimension":3},{"edge_id":"FTI-03","source_dynkin":[0,1],"token":-1,"target_dynkin":[0,2],"target_dimension":6},{"edge_id":"FTI-04","source_dynkin":[0,1],"token":-1,"target_dynkin":[1,0],"target_dimension":3},{"edge_id":"FTI-05","source_dynkin":[0,1],"token":1,"target_dynkin":[0,0],"target_dimension":1},{"edge_id":"FTI-06","source_dynkin":[0,1],"token":1,"target_dynkin":[1,1],"target_dimension":8},{"edge_id":"FTI-07","source_dynkin":[0,2],"token":-1,"target_dynkin":[0,3],"target_dimension":10},{"edge_id":"FTI-08","source_dynkin":[0,2],"token":-1,"target_dynkin":[1,1],"target_dimension":8},{"edge_id":"FTI-09","source_dynkin":[0,2],"token":1,"target_dynkin":[0,1],"target_dimension":3},{"edge_id":"FTI-10","source_dynkin":[0,2],"token":1,"target_dynkin":[1,2],"target_dimension":15},{"edge_id":"FTI-11","source_dynkin":[0,3],"token":1,"target_dynkin":[0,2],"target_dimension":6},{"edge_id":"FTI-12","source_dynkin":[1,0],"token":-1,"target_dynkin":[0,0],"target_dimension":1},{"edge_id":"FTI-13","source_dynkin":[1,0],"token":-1,"target_dynkin":[1,1],"target_dimension":8},{"edge_id":"FTI-14","source_dynkin":[1,0],"token":1,"target_dynkin":[0,1],"target_dimension":3},{"edge_id":"FTI-15","source_dynkin":[1,0],"token":1,"target_dynkin":[2,0],"target_dimension":6},{"edge_id":"FTI-16","source_dynkin":[1,1],"token":-1,"target_dynkin":[0,1],"target_dimension":3},{"edge_id":"FTI-17","source_dynkin":[1,1],"token":-1,"target_dynkin":[1,2],"target_dimension":15},{"edge_id":"FTI-18","source_dynkin":[1,1],"token":-1,"target_dynkin":[2,0],"target_dimension":6},{"edge_id":"FTI-19","source_dynkin":[1,1],"token":1,"target_dynkin":[0,2],"target_dimension":6},{"edge_id":"FTI-20","source_dynkin":[1,1],"token":1,"target_dynkin":[1,0],"target_dimension":3},{"edge_id":"FTI-21","source_dynkin":[1,1],"token":1,"target_dynkin":[2,1],"target_dimension":15},{"edge_id":"FTI-22","source_dynkin":[1,2],"token":-1,"target_dynkin":[0,2],"target_dimension":6},{"edge_id":"FTI-23","source_dynkin":[1,2],"token":1,"target_dynkin":[1,1],"target_dimension":8},{"edge_id":"FTI-24","source_dynkin":[2,0],"token":-1,"target_dynkin":[1,0],"target_dimension":3},{"edge_id":"FTI-25","source_dynkin":[2,0],"token":-1,"target_dynkin":[2,1],"target_dimension":15},{"edge_id":"FTI-26","source_dynkin":[2,0],"token":1,"target_dynkin":[1,1],"target_dimension":8},{"edge_id":"FTI-27","source_dynkin":[2,0],"token":1,"target_dynkin":[3,0],"target_dimension":10},{"edge_id":"FTI-28","source_dynkin":[2,1],"token":-1,"target_dynkin":[1,1],"target_dimension":8},{"edge_id":"FTI-29","source_dynkin":[2,1],"token":1,"target_dynkin":[2,0],"target_dimension":6},{"edge_id":"FTI-30","source_dynkin":[3,0],"token":-1,"target_dynkin":[2,0],"target_dimension":6}]')

def edge_key(row):
    return (
        tuple(row["source_dynkin"]),
        int(row["token"]),
        tuple(row["target_dynkin"]),
    )

edge_results = []
pair_decompositions = {}

for row in CERTIFIED_EDGES:
    source_dynkin = tuple(row["source_dynkin"])
    target_dynkin = tuple(row["target_dynkin"])
    token = int(row["token"])

    source_iw = dynkin_to_iweight(source_dynkin)
    token_iw = (1,0,0) if token == 1 else (1,1,0)
    target_iw = dynkin_to_iweight(target_dynkin)

    pair = (source_iw, token_iw)
    if pair not in pair_decompositions:
        pair_decompositions[pair] = ops.find_direct_sum([source_iw, token_iw])

    decomposition = pair_decompositions[pair]
    multiplicity = int(decomposition.get(target_iw, 0))
    target_dimension = int(ops.calc_dimension(target_iw))

    result = {
        "edge_id": row["edge_id"],
        "source_dynkin": list(source_dynkin),
        "source_iweight": list(source_iw),
        "token": token,
        "token_iweight": list(token_iw),
        "target_dynkin": list(target_dynkin),
        "target_iweight": list(target_iw),
        "expected_target_dimension": int(row["target_dimension"]),
        "pyclebsch_target_dimension": target_dimension,
        "multiplicity": multiplicity,
        "passed": (
            multiplicity == 1
            and target_dimension == int(row["target_dimension"])
        ),
    }
    edge_results.append(result)

failed_edges = [row for row in edge_results if not row["passed"]]
assert len(edge_results) == 30
assert not failed_edges, failed_edges
assert len(pair_decompositions) == 18

print("FUSION PASS: 30/30 certified edges independently verified.")
print("Unique source×token products:", len(pair_decompositions))
for pair, decomposition in sorted(pair_decompositions.items()):
    print(pair, "->", decomposition)


## Compute and orthogonality-check all unique CGC products

In [ ]:

# CPU calculation. Each unique product dimension is at most 45.
RUN_FULL_CGC = True

CGC_CHECKPOINT = Path("/content/Y4_PYCLEBSCH_CGC_CHECKPOINT.json")
cgc_results = {}

if CGC_CHECKPOINT.exists():
    cgc_results = json.loads(CGC_CHECKPOINT.read_text())

if RUN_FULL_CGC:
    ordered_pairs = sorted(pair_decompositions)
    for index, pair in enumerate(ordered_pairs, start=1):
        key = f"{pair[0]}x{pair[1]}"
        if cgc_results.get(key, {}).get("passed"):
            print(f"[{index:02d}/{len(ordered_pairs)}] cached {key}")
            continue

        print(f"[{index:02d}/{len(ordered_pairs)}] calculating {key}", flush=True)
        started = time.time()
        # calc_cgcs creates/reuses the package CGC_Data cache.
        cgc.calc_cgcs([pair[0], pair[1]])
        passed = bool(cgc.check_cgcs([pair[0], pair[1]]))
        elapsed = time.time() - started

        cgc_results[key] = {
            "source_iweight": list(pair[0]),
            "token_iweight": list(pair[1]),
            "passed": passed,
            "elapsed_s": elapsed,
        }
        CGC_CHECKPOINT.write_text(json.dumps(cgc_results, indent=2))
        assert passed, key

    assert len(cgc_results) == 18
    assert all(row["passed"] for row in cgc_results.values())
    print("CGC PASS: orthogonality verified for all 18 unique products.")
else:
    print("Full CGC calculation disabled; direct-sum fusion audit still passed.")


## Export the microscopic capability report

In [ ]:

report = {
    "schema_version": "2026-06-13-microscopic-audit-v1",
    "repositories": {
        "ymcirc": {
            "commit": "f2073f6bd61758a8ab56975ac53c5be340e8cffc",
            "public_truncations": {
                key: {str(k): v for k, v in value.items()}
                for key, value in current_truncations.items()
            },
            "magnetic_data": available_hamiltonian_data,
            "plaquette_data": available_plaquette_data,
            "supports_d3_publicly": (
                "d=3" in available_hamiltonian_data
                and "d=3" in available_plaquette_data
            ),
        },
        "pyclebsch": {
            "commit": "35e3926b07761da5bfbe573fd6af74e64a0c1a82",
            "fusion_edges_verified": sum(row["passed"] for row in edge_results),
            "fusion_edges_total": len(edge_results),
            "unique_products": len(pair_decompositions),
            "cgc_products_checked": len(cgc_results),
            "all_cgc_checks_passed": (
                bool(cgc_results)
                and all(row["passed"] for row in cgc_results.values())
            ),
        },
    },
    "required_irreps_dynkin": [list(x) for x in REQUIRED_DYNKIN],
    "required_irreps_iweight": [list(x) for x in REQUIRED_IWEIGHTS],
    "installed_y4_bitmap": {str(k): v for k, v in Y4_BITMAP.items()},
    "fusion_edge_results": edge_results,
    "cgc_results": cgc_results,
    "capability_verdict": {
        "representation_layer_reproducible_now": True,
        "all_30_fusion_edges_reproduced": True,
        "current_public_ymcirc_can_run_full_d3_y4_cube": False,
        "blocking_assets": [
            "d=3 physical plaquette/site-singlet state dictionary for Y4 truncation",
            "d=3 magnetic Hamiltonian box-term matrix elements for Y4 truncation",
            "d=3 circuit/ordering support for the enlarged truncation",
        ],
        "next_implementation_target": (
            "generate the missing d=3 Y4 physical-state and magnetic-element "
            "dictionaries from pyclebsch CGCs, then project the microscopic "
            "Hamiltonian onto the certified one-flux basis"
        ),
    },
}

REPORT_PATH = Path("/content/Y4_CURRENT_ENCODING_CAPABILITY_REPORT.json")
REPORT_PATH.write_text(json.dumps(report, indent=2))
print("Wrote", REPORT_PATH)
print(json.dumps(report["capability_verdict"], indent=2))



## Interpretation

A passing notebook establishes two things directly from the current public repositories:

- `pyclebsch` can independently reproduce the full representation-theory skeleton required by the fourth-order certificate;
- the released `ymcirc` data layer cannot yet run the complete \(d=3\), ten-irrep cube calculation because the necessary three-dimensional plaquette-state and magnetic-element dictionaries are not present.

This is not a dead end. It identifies the next coding target precisely: generate those two dictionaries from the verified `pyclebsch` Clebsch–Gordan data and connect them to the existing `LatticeStateEncoder`/magnetic-Hamiltonian interface.
